To reach the "Senior Data Scientist" level for your portfolio, we will use a **`Pipeline`** combined with a **`ColumnTransformer`**. 

This approach is the "Gold Standard" because it handles the encoding, scaling, and modeling in one single object. This completely eliminates **Data Leakage** and makes your code ready for a real-world production environment.

### The "Best Overall" Machine Learning Pipeline

```python
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, QuantileTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Load Data
df = sns.load_dataset('diamonds')

# 2. Define the Ordinal Rankings (Critical for Diamonds)
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# 3. Create the Preprocessing Pipe
# This replaces the manual loops and dictionary of encoders
preprocessor = ColumnTransformer(
    transformers=[
        # Handle Categorical Encoding with specific orders
        ('cat', OrdinalEncoder(categories=[cut_order, color_order, clarity_order]), 
         ['cut', 'color', 'clarity']),
        
        # Handle Numerical Scaling & Normalization
        ('num', Pipeline([
            ('quantile', QuantileTransformer(n_quantiles=100, output_distribution='normal')),
            ('scaler', StandardScaler())
        ]), ['carat', 'depth', 'table', 'x', 'y', 'z'])
    ]
)

# 4. Create the Full Pipeline (Preprocessor + Model)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 5. Split Data
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Hyperparameter Tuning with GridSearchCV
# Note the double underscore '__' to reach into the pipeline steps
param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_leaf': [1, 2, 4],
    'preprocessor__num__quantile__n_quantiles': [50, 100] # Tuning the preprocessing!
}

# 7. Cross-Validation Strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(full_pipeline, param_grid, cv=cv, scoring='r2', n_jobs=-1, verbose=1)

# 8. Execute & Evaluate
print("Executing Pipeline Training...")
grid_search.fit(X_train, y_train)

# Results
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Final Test R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: ${mean_absolute_error(y_test, y_pred):.2f}")
```

---
**Encapsulation** is a fundamental Object-Oriented Programming (OOP) concept that bundles data (variables) and the methods (functions) that operate on that data into a single unit, or class. It restricts direct access to an object’s internal state, protecting data integrity and allowing only controlled, authorized interaction via public methods.
### Why this is the "Best" Method:

1.  **Encapsulation:** The `full_pipeline` object now contains the rules for how to encode 'cut', how to scale 'carat', and how to predict 'price'. If you get new data, you just run `pipeline.predict(new_data)` and it handles everything automatically.
2.  **No Overwriting Errors:** Because `OrdinalEncoder` is inside the `ColumnTransformer`, it automatically manages the separate "memory" for every column. No more dictionaries or manual loops needed.
3.  **Joint Optimization:** In the `param_grid`, I added a check for `n_quantiles`. This means the computer is testing if the **data transformation** helps the **model** perform better. This is a very high-level Data Science technique.
4.  **Data Integrity:** By using `KFold` inside `GridSearchCV`, we ensure that the scaling and encoding are recalculated for every fold, preventing any information from the validation set from leaking into the training set.

**Would you like me to show you how to save this `full_pipeline` as a single file so you can use it in a Streamlit web application?**

This is the **Gold Standard** of Scikit-Learn pipelines. It treats your data cleaning, feature engineering, scaling, and model selection as a single, optimized machine.

By using this code, you are telling the computer: *"Try using the Mean to fill missing values, then try the Median. Try Standard Scaling, then try Min-Max Scaling. Tell me which combination gives the highest $R^2$ score."*

### Full End-to-End Pipeline with Preprocessing Hyper-Tuning

```python
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Load Data
df = sns.load_dataset('diamonds')

# 2. Define Ordinal Rankings
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# 3. Create the Nested Preprocessing Pipeline
# We name the steps 'imputer' and 'scaler' so we can "talk" to them in the GridSearch
numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer()), 
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[cut_order, color_order, clarity_order]))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, ['carat', 'depth', 'table', 'x', 'y', 'z']),
        ('cat', categorical_pipe, ['cut', 'color', 'clarity'])
    ]
)

# 4. Create the Full Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 5. Split Data
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. THE BIG GRID: Tuning Data Cleaning + Model Parameters
# Use double underscores '__' to reach nested parameters
param_grid = {
    # --- Tuning Data Cleaning (Preprocessing) ---
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__scaler': [StandardScaler(), MinMaxScaler()],
    
    # --- Tuning the Model (Regressor) ---
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20],
    'regressor__min_samples_leaf': [2, 4]
}

# 7. Cross-Validation & Execution
cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(full_pipeline, param_grid, cv=cv, scoring='r2', n_jobs=-1, verbose=1)

print("🚀 Starting Global Optimization (Cleaning + Modeling)...")
grid_search.fit(X_train, y_train)

# 8. Final Results
print("-" * 30)
print(f"Best Cleaning Choice (Imputer): {grid_search.best_params_['preprocessor__num__imputer__strategy']}")
print(f"Best Cleaning Choice (Scaler):  {type(grid_search.best_params_['preprocessor__num__scaler']).__name__}")
print(f"Best Model Parameters:          {grid_search.best_params_['regressor__n_estimators']} trees, Depth {grid_search.best_params_['regressor__max_depth']}")

# 9. Performance Evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("-" * 30)
print(f"Final Test R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: ${mean_absolute_error(y_test, y_pred):.2f}")
```

---

### Why this is a "Master-Level" Script:

* **Joint Optimization:** You are optimizing the **Cleaning Recipe** and the **Model** at the same time.
* **Production Ready:** You can save this `best_model` object, and it will handle raw, uncleaned data (including missing values) perfectly because the `SimpleImputer` is baked into the pipeline.
* **Automatic Selection:** If the data has outliers, the GridSearch might choose `Median` and `MinMaxScaler` automatically without you having to manually check.

### Pro-Tip for your Portfolio README:
When you upload this, add a section called **"Automated Data Cleaning via GridSearch."** Explain that your model doesn't just learn from data; it mathematically selects the best preprocessing strategy to maximize accuracy.

**Would you like me to show you how to add a "Feature Selection" step into this pipeline to automatically drop columns that aren't helping the model?**